In [4]:
import json
import random
import string
import urllib.request
import urllib.error
from datetime import date
from pathlib import Path

# Saving files in the local directory
DATA_FILE = Path("cryptoquote_data.json")

# Sourcing quotes from Zenquotes API (which has a random query function)
QUOTE_API = "https://zenquotes.io/api/random"

def get_cryptoquote(quote):
    """Using a random substitution cipher to encrypt a given quote"""
    # Convert to uppercase
    quote = quote.upper()

    # Create a list of unique characters in the quote
    unique_chars = list(set(quote))
    for char in unique_chars: # Remove if removing punctuation
        if char in string.punctuation:
            unique_chars.remove(char)
    if ' ' in unique_chars:
        unique_chars.remove(' ')

    # Create a shuffled version of the unique characters for the substitution
    shuffled_chars = unique_chars[:]
    while True:
        random.shuffle(shuffled_chars)
        # Ensure that no character maps to itself
        if all(uc != sc for uc, sc in zip(unique_chars, shuffled_chars)):
            break
    
    # Create the substitution dictionary
    substitution_dict = {uc: sc for uc, sc in zip(unique_chars, shuffled_chars)}
    
    # Generate the cryptoquote
    cryptoquote = ''.join([substitution_dict[char] if char in substitution_dict else char for char in quote])
    
    return cryptoquote, substitution_dict


def fetch_quote():
    """Fetching a random quote and its author from ZenQuotes."""
    request = urllib.request.Request(
        QUOTE_API,
        headers={"User-Agent": "CryptoquoteOfTheDay/1.0"}
    )

    with urllib.request.urlopen(request, timeout=15) as response:
        data = json.loads(response.read().decode("utf-8"))

    # Controlling for API errors
    if not data or not data[0].get("q"):
        raise ValueError("The quote API returned no usable quote.")

    quote = data[0]["q"].strip()
    author = data[0].get("a", "Unknown").strip()

    return quote, author


def load_daily_puzzle():
    """Loading the puzzle of the day or creating one (if the game is loading for the first time)"""
    today = date.today().isoformat()

    # Loading saved data if it exists
    if DATA_FILE.exists():
        try:
            with DATA_FILE.open("r", encoding="utf-8") as file:
                saved_data = json.load(file)

            if saved_data.get("date") == today:
                return saved_data

        except (json.JSONDecodeError, OSError):
            print("Could not read saved data; creating a new puzzle.")

    # Requesting a quote until the API returns one long enough to be played
    for _ in range(5):
        quote, author = fetch_quote()
        quote = quote.upper()

        # Limiting the game to quotes 15 characters long or over (in order for the quote to be solvable)
        if sum(char in string.ascii_uppercase for char in quote) >= 15:
            break
    else:
        raise ValueError("Could not find a long enough quote.")

    cryptoquote, substitution_dict = get_cryptoquote(quote)

    puzzle = {
        "date": today,
        "quote": quote,
        "author": author,
        "cryptoquote": cryptoquote,
        "substitution": substitution_dict
    }

    # Saving the solution locally to prevent infinite reloading (querying the API unlimited times)
    with DATA_FILE.open("w", encoding="utf-8") as file:
        json.dump(puzzle, file, indent=2, ensure_ascii=False)

    return puzzle


def show_puzzle(puzzle):
    """Displaying the encrypted puzzle"""
    print(f"CRYPTOQUOTE OF THE DAY ({puzzle['date']})")
    print()
    print("Today's quotation:")
    print()
    print(puzzle["cryptoquote"])
    print()
    print("-" * 60)
    print("Source: ZenQuotes (zenquotes.io)") # Attribution required to query the API
    print("-" * 60)
    print("Commands:")
    print("HINT - Reveal one letter mapping")
    print("SOLVE - Reveal the answer")
    print("QUIT - Exit the game")
    print()

    while True:
        command = input("Command input box: ").strip().upper()

        if command == "QUIT":
            print("Bye bye!")
            break

        elif command == "SOLVE":
            print()
            print("Answer:")
            print(puzzle["quote"])
            print(f"— {puzzle['author']}")
            break

        elif command == "HINT":
            mappings = list(puzzle["substitution"].items())

            # Reveal a new letter mapping
            available = [
                pair for pair in mappings
                if pair[0] not in puzzle.get("revealed", []) # Preventing us from showing letter mappings that have
                                                            # Already been revealed
            ]

            if not available:
                print("All of the letter mappings have already been revealed!") # Preventing infinite hints
                continue

            original, cipher = random.choice(available)
            puzzle.setdefault("revealed", []).append(original)

            print(f"Hint: {cipher} represents {original}")

            # Saving revealed hints in the save data
            with DATA_FILE.open("w", encoding="utf-8") as file:
                json.dump(puzzle, file, indent=2, ensure_ascii=False)

        else:
            print("Please enter HINT, SOLVE, or QUIT.")


def main():
    try:
        puzzle = load_daily_puzzle()
        show_puzzle(puzzle)

    except (urllib.error.URLError, TimeoutError, ValueError) as error:
        print(f"Could not load today's puzzle: {error}")
        print("Check your internet connection and try again.")

    except OSError as error:
        print(f"File error: {error}")


if __name__ == "__main__":
    main()

CRYPTOQUOTE OF THE DAY (2026-10-17)

Today's quotation:

UB UO'H WEO FDE,TWR OUW,TD YUOK UO OUII LEN BUWS ENO KEY UO YED,H.

------------------------------------------------------------
Source: ZenQuotes (zenquotes.io)
------------------------------------------------------------
Commands:
HINT - Reveal one letter mapping
SOLVE - Reveal the answer
QUIT - Exit the game



Command input box:  solve



Answer:
IF IT'S NOT BROKEN, TINKER WITH IT TILL YOU FIND OUT HOW IT WORKS.
— Bob Proctor
